## *1. Problem Overview*

Building energy efficiency modeling is a critical domain in computational sustainability and urban energy management. Commercial and residential facilities contribute significantly to regional greenhouse gas emissions and electrical grid loads. Accurately modeling Site Energy Use Intensity (`site_eui`), expressed in kBtu/sq.ft, allows municipal planners, building managers, and policy makers to benchmark energy performance, identify energy-inefficient facilities, and prioritize retrofitting projects across diverse property types.


### *Problem Statement*

Given a comprehensive dataset of commercial and residential building characteristics, climate variables, and historical temperature metrics across various states, the challenge is to formulate a robust predictive regression framework. The model must accurately estimate `site_eui` based on facility attributes (such as `floor_area`, `year_built`, `energy_star_rating`, and `facility_type`) and local climatic indicators (such as heating and cooling degree days), despite high right-skewness, missing data, and potential linear dependency among predictors.


### *Regression Objective*

The objective of this project is to construct an end-to-end regression modeling pipeline to predict building Site Energy Use Intensity (`site_eui`). The pipeline covers raw dataset auditing, missing value treatment, duplicate removal, exploratory data analysis (EDA), domain feature engineering, zero-leakage preprocessing, and training, hyperparameter tuning, and comparative evaluation across all 10 required regression algorithms (Linear Regression, Ridge Regression, Lasso Regression, ElasticNet Regression, Polynomial Regression, Decision Tree Regressor, Random Forest Regressor, Gradient Boosting Regressor, Support Vector Regression, and K-Nearest Neighbors). All models are trained and evaluated on a unified stratified 80:20 train-test split using $R^2$, Root Mean Squared Error (RMSE), and Mean Absolute Error (MAE) metrics, with 5-fold cross-validation applied to the top-performing models, culminating in a single consolidated model comparison table as required by the Review 1 Capstone rubric.


## *2. Import Required Libraries*

We import core Python data manipulation libraries (`pandas`, `numpy`), visualization frameworks (`matplotlib.pyplot`, `seaborn`), and scikit-learn modules for train-test splitting, feature preprocessing, baseline linear models, grid search tuning, regression metrics, and model persistence via `joblib`.

- **Data Manipulation:** `pandas` for DataFrame manipulation, `numpy` for log/exp transformations and vector math.
- **Visualization:** `matplotlib.pyplot` and `seaborn` (configured with a **colorblind-friendly color palette**) for distribution plots, correlation heatmaps, scatter plots, and diagnostic residual plots.
- **Preprocessing & Persistence:** `StandardScaler`, `OneHotEncoder`, `PolynomialFeatures`, `SimpleImputer`, `ColumnTransformer`, and `joblib` to assemble and export leakage-free pipeline objects.
- **Regression Algorithms:** `LinearRegression`, `Ridge`, `Lasso`, and `ElasticNet` for baseline linear algorithms.
- **Model Evaluation:** `r2_score`, `mean_squared_error`, `mean_absolute_error` for rigorous performance evaluation.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Set global formatting and colorblind-friendly plotting styles (Rubric A2)
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 10
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print("Libraries successfully imported and global colorblind-friendly style initialized.")


Libraries successfully imported and global colorblind-friendly style initialized.


## *3. Dataset Loading*


We load the raw building dataset (`train.csv`) into a pandas DataFrame.


In [2]:
df_raw = pd.read_csv('train.csv')
print("Raw dataset loaded successfully.")


Raw dataset loaded successfully.


We check the dataset dimensions (rows and columns) in an isolated cell (Rubric A1).


In [3]:
print("Dataset Shape:", df_raw.shape)


Dataset Shape: (75757, 64)


We inspect column data types in an isolated cell with a clean tabular summary (Rubric A1).


In [4]:
df_dtypes_summary = pd.DataFrame({
    'Data Type': df_raw.dtypes.value_counts().index.astype(str),
    'Column Count': df_raw.dtypes.value_counts().values
})
print("Column Data Types Summary:")
display(df_dtypes_summary)


Column Data Types Summary:


,Data Type,Column Count
0,int64,37
1,float64,24
2,object,3


We audit missing-value counts per column in an isolated cell (Rubric A1).


In [5]:
missing_counts_audit = df_raw.isnull().sum()
missing_counts_audit = missing_counts_audit[missing_counts_audit > 0].sort_values(ascending=False)
print("=== MISSING VALUE COUNTS PER COLUMN ===")
print(missing_counts_audit)


=== MISSING VALUE COUNTS PER COLUMN ===
days_with_fog                45796
direction_peak_wind_speed    41811
direction_max_wind_speed     41082
max_wind_speed               41082
energy_star_rating           26709
year_built                    1837
dtype: int64


We compute summary distribution statistics for the target variable `site_eui` in an isolated cell (Rubric A1).


In [6]:
df_target_stats = pd.DataFrame(df_raw['site_eui'].describe()).reset_index()
df_target_stats.columns = ['Statistic', 'site_eui (kBtu/sq.ft)']
df_target_stats['site_eui (kBtu/sq.ft)'] = df_target_stats['site_eui (kBtu/sq.ft)'].round(2)

print("=== TARGET VARIABLE ('site_eui') DISTRIBUTION SUMMARY STATS ===")
display(df_target_stats)


=== TARGET VARIABLE ('site_eui') DISTRIBUTION SUMMARY STATS ===


,Statistic,site_eui (kBtu/sq.ft)
0,count,75757.00
1,mean,82.58
2,std,58.26
3,min,1.00
4,25%,54.53
5,50%,75.29
6,75%,97.28
7,max,997.87


We display the first five records of the dataset.


In [7]:
df_raw.head()


,Year_Factor,State_Factor,building_class,facility_type,floor_area,year_built,energy_star_rating,ELEVATION,january_min_temp,january_avg_temp,...,days_above_80F,days_above_90F,days_above_100F,days_above_110F,direction_max_wind_speed,direction_peak_wind_speed,max_wind_speed,days_with_fog,site_eui,id
0,1,State_1,Commercial,Grocery_store_or_food_market,61242.0,1942.0,11.0,2.4,36,50.5,...,14,0,0,0,1.0,1.0,1.0,NaN,248.682615,0
1,1,State_1,Commercial,Warehouse_Distribution_or_Shipping_center,274000.0,1955.0,45.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,26.500150,1
2,1,State_1,Commercial,Retail_Enclosed_mall,280025.0,1951.0,97.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,24.693619,2
3,1,State_1,Commercial,Education_Other_classroom,55325.0,1980.0,46.0,1.8,36,50.5,...,14,0,0,0,1.0,NaN,1.0,12.0,48.406926,3
4,1,State_1,Commercial,Warehouse_Nonrefrigerated,66000.0,1985.0,100.0,2.4,36,50.5,...,14,0,0,0,1.0,1.0,1.0,NaN,3.899395,4


We display the last five records of the dataset.


In [8]:
df_raw.tail()


,Year_Factor,State_Factor,building_class,facility_type,floor_area,year_built,energy_star_rating,ELEVATION,january_min_temp,january_avg_temp,...,days_above_80F,days_above_90F,days_above_100F,days_above_110F,direction_max_wind_speed,direction_peak_wind_speed,max_wind_speed,days_with_fog,site_eui,id
75752,6,State_11,Commercial,Office_Uncategorized,20410.0,1995.0,8.0,36.6,28,43.451613,...,25,3,0,0,NaN,NaN,NaN,NaN,132.918411,75752
75753,6,State_11,Residential,5plus_Unit_Building,40489.0,1910.0,98.0,36.6,28,43.451613,...,25,3,0,0,NaN,NaN,NaN,NaN,39.483672,75753
75754,6,State_11,Commercial,Commercial_Other,28072.0,1917.0,NaN,36.6,26,36.612903,...,6,0,0,0,NaN,NaN,NaN,NaN,48.404398,75754
75755,6,State_11,Commercial,Commercial_Other,53575.0,2012.0,NaN,36.6,26,36.612903,...,6,0,0,0,NaN,NaN,NaN,NaN,592.022750,75755
75756,6,State_11,Residential,2to4_Unit_Building,23888.0,1974.0,51.0,36.6,27,36.935484,...,16,0,0,0,NaN,NaN,NaN,NaN,29.154684,75756


We print the complete list of column names in the raw dataset.


In [9]:
print("Dataset Column Names:\n", df_raw.columns.tolist())


Dataset Column Names:
 ['Year_Factor', 'State_Factor', 'building_class', 'facility_type', 'floor_area', 'year_built', 'energy_star_rating', 'ELEVATION', 'january_min_temp', 'january_avg_temp', 'january_max_temp', 'february_min_temp', 'february_avg_temp', 'february_max_temp', 'march_min_temp', 'march_avg_temp', 'march_max_temp', 'april_min_temp', 'april_avg_temp', 'april_max_temp', 'may_min_temp', 'may_avg_temp', 'may_max_temp', 'june_min_temp', 'june_avg_temp', 'june_max_temp', 'july_min_temp', 'july_avg_temp', 'july_max_temp', 'august_min_temp', 'august_avg_temp', 'august_max_temp', 'september_min_temp', 'september_avg_temp', 'september_max_temp', 'october_min_temp', 'october_avg_temp', 'october_max_temp', 'november_min_temp', 'november_avg_temp', 'november_max_temp', 'december_min_temp', 'december_avg_temp', 'december_max_temp', 'cooling_degree_days', 'heating_degree_days', 'precipitation_inches', 'snowfall_inches', 'snowdepth_inches', 'avg_temp', 'days_below_30F', 'days_below_20F', 